<a href="https://colab.research.google.com/github/AlperYildirim1/Language-as-Waves/blob/main/FNet_Train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q torchmetrics sacrebleu x-transformers

## CONFIG

In [ ]:
!pip install -q torchmetrics sacrebleu x-transformers

## CONFIG

# --- Data & Task Size ---
MAX_LENGTH = 128

MODEL_CHOICE = "FFNet-A100-80GB-v4-high-precision-fixed" # Renamed for clarity

# --- Model Architecture Config ---
D_MODEL = 512
NUM_HEADS = 8
D_FF = 2048
DROPOUT = 0.1

# --- Layer counts ---
NUM_ENCODER_LAYERS = 7
NUM_DECODER_LAYERS = 6

# --- Training Config (ADJUSTED FOR FAIR COMPARISON) ---

TARGET_TRAINING_STEPS = 100000
GRAD_ACCUMULATION_STEPS = 2


VALIDATION_SCHEDULE = [
    2000, 4000, 5000, 7500, 10000, 15000, 20000,
    25000, 30000, 35000, 42500, 50000, 57500, 65000, 72500, 90000, 100000
]
PEAK_LEARNING_RATE = 6e-4
WARMUP_STEPS = 600 # Warmup can stay similar or scale slightly, 600 is fine
WEIGHT_DECAY = 0.01

# --- Regularization Config ---
LABEL_SMOOTHING_EPSILON = 0.1

# --- Other Constants ---
DRIVE_BASE_PATH = "/content/drive/MyDrive/AIAYN"
ORIGINAL_BUCKETED_REPO_ID = "Yujivus/wmt14-de-en-bucketed-w4" # Use the bucketed one (we will ignore buckets)
MODEL_CHECKPOINT = "Helsinki-NLP/opus-mt-de-en"

## DATALOADERS

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from datasets import load_dataset
import math
import os
from tqdm.auto import tqdm
from torch.utils.tensorboard import SummaryWriter
import random
import numpy as np
import torch
from transformers import get_cosine_schedule_with_warmup
from typing import List
from transformers import AutoModel
from transformers import DataCollatorForSeq2Seq


def set_seed(seed_value=5):
    """Sets the seed for reproducibility."""
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 117
set_seed(SEED)
print(f"Reproducibility seed set to {SEED}")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

#torch.use_deterministic_algorithms(True)

print("--- Loading Modernized Configuration ---")
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

torch.set_float32_matmul_precision('high')
print("✅ PyTorch matmul precision set to 'high'")

# --- Device Setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

VOCAB_SIZE = len(tokenizer)
print(f"Vocab size: {VOCAB_SIZE}")


# DATA LOADING & PREPARATION

# --- 1. DEFINE THE FNET COLLATOR (FORCE FIXED LENGTH) ---
# This is crucial. It forces every sentence to be exactly 128 tokens.
fnet_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding="max_length",    # <--- FORCE PADDING
    max_length=MAX_LENGTH,   # <--- 128 (defined in your config)
    pad_to_multiple_of=None
)

# --- 2. LOAD DATASET ---
print(f"Loading original bucketed samples from: {ORIGINAL_BUCKETED_REPO_ID}")
original_datasets = load_dataset(ORIGINAL_BUCKETED_REPO_ID)

# --- 3. CREATE DATALOADERS (STANDARD FIXED SIZE) ---
FNET_PHYSICAL_BATCH_SIZE = 320

g = torch.Generator()
g.manual_seed(SEED)

train_dataloader = DataLoader(
    original_datasets["train"],
    batch_size=FNET_PHYSICAL_BATCH_SIZE,  # <--- FIXED BATCH SIZE (Safe from OOM)
    shuffle=True,                # <--- GLOBAL SHUFFLE
    num_workers=8,
    collate_fn=fnet_collator,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=g,
)

val_dataloader = DataLoader(
    original_datasets["validation"],
    batch_size=FNET_PHYSICAL_BATCH_SIZE,
    collate_fn=fnet_collator,
    num_workers=8,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=g,
)

print(f"Train Dataloader is now a STANDARD iterator.")
print(f"Physical Batch Size: {FNET_PHYSICAL_BATCH_SIZE}")
print(f"Gradient Accumulation: {GRAD_ACCUMULATION_STEPS}")
print(f"Effective Batch Size: {FNET_PHYSICAL_BATCH_SIZE * GRAD_ACCUMULATION_STEPS}")

# --- SANITY CHECK ---
print("\n--- Running Sanity Check on new FNet DataLoader ---")
train_dataloader.generator.manual_seed(SEED)
temp_iterator = iter(train_dataloader)
print("Shapes of first 3 batches (Should all be [64, 128]):")
for i in range(3):
    batch = next(temp_iterator)
    print(f"  Batch {i+1}: input_ids shape = {batch['input_ids'].shape}")
print("--- Sanity Check Complete ---\n")
# --- VERIFY SHUFFLE IS WORKING ---
print("🕵️ INSPECTING ONE BATCH 🕵️")

# Get one batch from your active train_dataloader
batch = next(iter(train_dataloader))
input_ids = batch['input_ids']

# Calculate real lengths (ignoring padding)
# We count how many tokens are NOT the pad token (usually 0 or 58100)
real_lengths = (input_ids != tokenizer.pad_token_id).sum(dim=1)

print(f"Batch Shape: {input_ids.shape}")
print("Random Sample of 20 lengths in this batch:")
print(real_lengths[:20].tolist())

# Check diversity
if real_lengths.float().std() < 5:
    print("\n⚠️ WARNING: LENGTHS LOOK CLUSTERED! (Bad shuffling)")
else:
    print(f"\n✅ PASSED: Lengths are highly variable (Std Dev: {real_lengths.float().std():.2f}). Shuffling is working.")

##  Models

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from x_transformers import Encoder, Decoder

# ==============================================================================
# 2. IMPORTS & SETUP
# ==============================================================================
from x_transformers import Decoder
# --- NETWORK FIX: INCREASE TIMEOUT ---
import socket
import os

# 1. Increase the global socket timeout (default is usually too short for large downloads)
socket.setdefaulttimeout(300)  # Set to 5 minutes

# 2. explicit retry settings for Hugging Face
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1" # Fast download (optional, but helps)
def set_seed(seed_value=116):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Logging Setup ---
experiment_name = f"{MODEL_CHOICE}_{datetime.datetime.now().strftime('%Y%m%d_%H%M')}"
CURRENT_RUN_DIR = os.path.join(DRIVE_BASE_PATH, experiment_name)
SAVE_DIR = os.path.join(CURRENT_RUN_DIR, "models")
LOG_DIR_TENSORBOARD = os.path.join(CURRENT_RUN_DIR, "tensorboard_logs")
LOG_FILE_TXT = os.path.join(CURRENT_RUN_DIR, "run_log.txt")

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(LOG_DIR_TENSORBOARD, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(message)s',
    handlers=[logging.FileHandler(LOG_FILE_TXT), logging.StreamHandler(sys.stdout)],
    force=True
)
logger = logging.getLogger(__name__)
writer = SummaryWriter(LOG_DIR_TENSORBOARD)

# ==============================================================================
# 3. DATA LOADING
# ==============================================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
VOCAB_SIZE = len(tokenizer)
standard_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer)

class PreBatchedCollator:
    def __init__(self, original_dataset_split):
        self.original_dataset = original_dataset_split
    def __call__(self, features: List[dict]) -> dict:
        batch_indices = features[0]['batch_indices']
        dict_of_lists = self.original_dataset[batch_indices]
        list_of_dicts = []
        keys = dict_of_lists.keys()
        num_samples = len(dict_of_lists['input_ids'])
        for i in range(num_samples):
            list_of_dicts.append({key: dict_of_lists[key][i] for key in keys})
        return standard_collator(list_of_dicts)

logger.info(f"Loading datasets...")
prebatched_datasets = load_dataset(PREBATCHED_REPO_ID)
original_datasets = load_dataset(ORIGINAL_BUCKETED_REPO_ID)
train_collator = PreBatchedCollator(original_datasets["train"])

train_dataloader = DataLoader(
    prebatched_datasets["train"], batch_size=1, shuffle=True,
    collate_fn=train_collator, num_workers=2, pin_memory=True, prefetch_factor=2
)
val_dataloader = DataLoader(
    original_datasets["validation"], batch_size=64,
    collate_fn=standard_collator, num_workers=2
)
# ==============================================================================
# 4. PRISM ARCHITECTURE (FIXED: COMPLEX DROPOUT & PADDING)
# ==============================================================================

# ==============================================================================
# 4. PRISM ARCHITECTURE (CLEAN & CORRECTED)
# ==============================================================================

class ComplexDropout(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        self.p = p

    def forward(self, z):
        if not self.training or self.p == 0.0:
            return z
        mask = torch.ones_like(z.real)
        mask = F.dropout(mask, self.p, self.training, inplace=False)
        return z * mask

class PhasePreservingLayerNorm(nn.Module):
    def __init__(self, d_model, eps=1e-5):
        super().__init__()
        self.layernorm = nn.LayerNorm(d_model, eps=eps)
        self.eps = eps

    def forward(self, x):
        mag = torch.abs(x)
        mag_norm = self.layernorm(mag)
        return mag_norm.to(x.dtype) * (x / (mag + self.eps))

class HarmonicEmbedding(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, max_period=10000.0):
        super().__init__()
        self.embedding_dim = embedding_dim

        # 1. MASTER REAL EMBEDDING (The "Source of Truth")
        # Shared by Encoder (before projection) and Decoder (directly)
        self.raw_embedding = nn.Embedding(num_embeddings, embedding_dim)

        # 2. THE ADAPTER (The "Projection")
        # Projects Real (D) -> Complex Components (2D)
        self.adapter = nn.Linear(embedding_dim, embedding_dim * 2)

        # Frequency logic (unchanged)
        freqs = torch.exp(torch.arange(0, embedding_dim, dtype=torch.float32) * -(math.log(max_period) / embedding_dim))
        self.register_buffer('freqs', freqs)

    def forward(self, input_ids):
        # A. Lookup Real Vector
        real_base = self.raw_embedding(input_ids) # [Batch, Seq, D]

        # B. Project to 2D for Complex Construction
        complex_params = self.adapter(real_base)  # [Batch, Seq, 2D]

        # C. Split into Real/Imag
        real = complex_params[..., :self.embedding_dim]
        imag = complex_params[..., self.embedding_dim:]

        # D. Construct Complex Number
        content_z = torch.complex(real, imag)

        # E. Apply RoPE (Rotary Position Embedding)
        seq_len = input_ids.shape[1]
        positions = torch.arange(seq_len, device=input_ids.device).float()
        angles = torch.outer(positions, self.freqs)
        pos_rotation = torch.polar(torch.ones_like(angles), angles).unsqueeze(0)

        return content_z * pos_rotation


class ModReLU(nn.Module):
    def __init__(self, features):
        super().__init__()
        self.b = nn.Parameter(torch.zeros(features))
    def forward(self, z):
        mag = torch.abs(z)
        new_mag = F.relu(mag + self.b)
        phase = z / (mag + 1e-6)
        return new_mag * phase

# --- THE CORRECT LAYER (Cartesian Gated) ---
class PRISMLayer(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.filter_len = max_len

        # 1. THE GATE (Data Dependency)
        self.gate_proj = nn.Linear(d_model * 2, d_model * 2)

        # 2. THE FILTER (Global Pattern)
        self.global_filter = nn.Parameter(torch.randn(d_model, max_len, dtype=torch.cfloat) * 0.02)

        # 3. INPUT MIXING
        self.mix_real = nn.Linear(d_model, d_model)
        self.mix_imag = nn.Linear(d_model, d_model)

        # 4. OUTPUT PROJECTION
        self.out_real = nn.Linear(d_model, d_model)
        self.out_imag = nn.Linear(d_model, d_model)

        self.activation = ModReLU(d_model)
        self.norm = PhasePreservingLayerNorm(d_model)
        self.dropout = ComplexDropout(dropout)

    def complex_linear(self, x, l_real, l_imag):
        r, i = x.real, x.imag
        new_r = l_real(r) - l_imag(i)
        new_i = l_real(i) + l_imag(r)
        return torch.complex(new_r, new_i)

    def forward(self, x, src_mask=None):
        if x is None: return None
        residual = x
        x_norm = self.norm(x)

        if src_mask is not None:
            x_norm = x_norm.masked_fill(src_mask.unsqueeze(-1), 0.0)

        # A. GATE
        x_cat = torch.cat([x_norm.real, x_norm.imag], dim=-1)
        gates = torch.sigmoid(self.gate_proj(x_cat))
        gate_r, gate_i = gates.chunk(2, dim=-1)

        # B. FILTER
        B, L, D = x_norm.shape
        x_freq = torch.fft.fft(x_norm, n=self.filter_len, dim=1)
        x_filtered = x_freq * self.global_filter.transpose(-1, -2)
        x_time = torch.fft.ifft(x_filtered, n=self.filter_len, dim=1)
        x_time = x_time[:, :L, :]

        # C. APPLY GATE
        gated_r = x_time.real * gate_r
        gated_i = x_time.imag * gate_i
        x_gated = torch.complex(gated_r, gated_i)

        # D. OUT
        x_mixed = self.complex_linear(x_gated, self.mix_real, self.mix_imag)
        x_act = self.activation(x_mixed)
        out = self.complex_linear(x_act, self.out_real, self.out_imag)
        return self.dropout(out) + residual

# --- ENCODER MUST BE DEFINED AFTER LAYER ---

class PRISMEncoder(nn.Module):
    def __init__(self, num_layers, d_model, max_len, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([PRISMLayer(d_model, max_len, dropout) for _ in range(num_layers)])
        self.final_norm = PhasePreservingLayerNorm(d_model)

    def forward(self, x, src_mask=None):
        for layer in self.layers:
            # EĞER TRAINING MODUNDAYSA CHECKPOINT KULLAN
            if self.training:
                # use_reentrant=False modern ve güvenli olandır
                x = torch.utils.checkpoint.checkpoint(layer, x, src_mask, use_reentrant=False)
            else:
                x = layer(x, src_mask)
        return self.final_norm(x)

# --- THE CORRECT BRIDGE (Cartesian) ---
class ComplexToRealBridge(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.proj = nn.Linear(d_model * 2, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x_complex):
        if x_complex is None: raise ValueError("Bridge None")
        cat = torch.cat([x_complex.real, x_complex.imag], dim=-1)
        return self.norm(self.proj(cat))

class PRISMHybrid_RoPE(nn.Module):
    def __init__(self, num_encoder_layers, num_refining_layers, num_decoder_layers,
                 num_heads, d_model, dff, vocab_size, max_length, dropout):
        super().__init__()

        self.d_model = d_model
        # 1. Initialize HarmonicEmbedding (It now holds the Master Embedding)
        self.harmonic_embedding = HarmonicEmbedding(vocab_size, d_model)

        # 2. POINTER MAGIC: Point the decoder's embedding to the Encoder's Master
        self.tgt_embedding = self.harmonic_embedding.raw_embedding

        self.dropout = nn.Dropout(dropout)

        if num_encoder_layers > 0:
            self.prism_encoder = PRISMEncoder(num_encoder_layers, d_model, max_length, dropout)
        else:
            self.prism_encoder = None

        self.bridge = ComplexToRealBridge(d_model)

        if num_refining_layers > 0:
            refining_layer = nn.TransformerEncoderLayer(
                d_model, num_heads, dff, dropout,
                batch_first=True, norm_first=True
            )
            self.reasoning_encoder = nn.TransformerEncoder(refining_layer, num_layers=num_refining_layers)
        else:
            self.reasoning_encoder = None

        self.decoder = Decoder(
            dim = d_model, depth = num_decoder_layers, heads = num_heads, attn_dim_head = d_model // num_heads,
            ff_mult = dff / d_model, rotary_pos_emb = True, cross_attend = True, attn_flash = True,
            attn_dropout = dropout, ff_dropout = dropout, use_rmsnorm = True
        )

        # 3. Output Projection
        self.final_linear = nn.Linear(d_model, vocab_size)

        # 4. WEIGHT TYING (The "Press Attention Later" requirement)
        # We tie the Output Head weights to the Master Input Embedding weights
        self.final_linear.weight = self.tgt_embedding.weight

    def create_masks(self, src, tgt):
        src_padding_mask = (src == tokenizer.pad_token_id)
        tgt_padding_mask = (tgt == tokenizer.pad_token_id)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(sz=tgt.size(1), device=src.device, dtype=torch.bool)
        return src_padding_mask, tgt_padding_mask, src_padding_mask, tgt_mask

    def forward(self, src, tgt, src_mask, tgt_pad, mem_pad, tgt_mask):
        src_harmonic = self.harmonic_embedding(src)
        if src_mask is not None:
            src_harmonic = src_harmonic.masked_fill(src_mask.unsqueeze(-1), 0.0)

        if self.prism_encoder is not None:
            # Checkpointing artık PRISMEncoder içinde katman-bazlı yapılıyor.
            # Burada tekrar checkpoint çağırmaya gerek yok (Hata verdirir).
            encoded_complex = self.prism_encoder(src_harmonic, src_mask)
        else:
            encoded_complex = src_harmonic

        coarse_memory = self.bridge(encoded_complex)
        if self.reasoning_encoder is not None:
            refined_memory = self.reasoning_encoder(coarse_memory, src_key_padding_mask=mem_pad)
        else:
            refined_memory = coarse_memory

        tgt_emb = self.tgt_embedding(tgt) * math.sqrt(self.d_model)
        tgt_emb = self.dropout(tgt_emb)
        context_mask = ~mem_pad if mem_pad is not None else None
        decoder_mask = ~tgt_pad if tgt_pad is not None else None

        if self.training:
            tgt_emb.requires_grad_(True)
            output = torch.utils.checkpoint.checkpoint(
                self.decoder, tgt_emb, context=refined_memory, mask=decoder_mask, context_mask=context_mask, use_reentrant=False
            )
        else:
            output = self.decoder(tgt_emb, context=refined_memory, mask=decoder_mask, context_mask=context_mask)

        return self.final_linear(output)

    # ... (generate function remains the same) ...
    @torch.no_grad()
    def generate(self, src, max_length, num_beams=5):
        self.eval()
        src_mask = (src == tokenizer.pad_token_id)
        context_mask = ~src_mask
        src_harmonic = self.harmonic_embedding(src)
        if src_mask is not None:
            src_harmonic = src_harmonic.masked_fill(src_mask.unsqueeze(-1), 0.0)

        if self.prism_encoder is not None:
            encoded_complex = self.prism_encoder(src_harmonic, src_mask)
        else:
            encoded_complex = src_harmonic

        coarse_memory = self.bridge(encoded_complex)

        if self.reasoning_encoder is not None:
            memory = self.reasoning_encoder(coarse_memory, src_key_padding_mask=src_mask)
        else:
            memory = coarse_memory

        batch_size = src.shape[0]
        memory = memory.repeat_interleave(num_beams, dim=0)
        context_mask = context_mask.repeat_interleave(num_beams, dim=0)

        beams = torch.full((batch_size * num_beams, 1), tokenizer.pad_token_id, dtype=torch.long, device=src.device)
        beam_scores = torch.zeros(batch_size * num_beams, device=src.device)
        finished_beams = torch.zeros(batch_size * num_beams, dtype=torch.bool, device=src.device)

        for _ in range(max_length - 1):
            if finished_beams.all(): break
            tgt_emb = self.tgt_embedding(beams) * math.sqrt(self.d_model)
            tgt_emb = self.dropout(tgt_emb)

            # Decoder
            decoder_output = self.decoder(tgt_emb, context=memory, context_mask=context_mask)
            logits = self.final_linear(decoder_output[:, -1, :])
            log_probs = F.log_softmax(logits, dim=-1)

            # Masking
            log_probs[:, tokenizer.pad_token_id] = -torch.inf
            if finished_beams.any(): log_probs[finished_beams, tokenizer.eos_token_id] = 0

            # --- BEAM SEARCH LOGIC FIX ---
            if _ == 0:
                # First Step: Expand from the first beam only (since all are identical start tokens)
                # Reshape to (batch, beams, vocab)
                total = (beam_scores.unsqueeze(1) + log_probs).view(batch_size, num_beams, -1)
                # Mask out all beams except the first one (-inf)
                total[:, 1:, :] = -torch.inf
                # Flatten back to (batch, beams*vocab) to pick top k
                total = total.view(batch_size, -1)
            else:
                # Subsequent Steps: Standard Flatten
                total = (beam_scores.unsqueeze(1) + log_probs).view(batch_size, -1)

            top_scores, top_indices = torch.topk(total, k=num_beams, dim=1)

            beam_indices = top_indices // log_probs.shape[-1]
            token_indices = top_indices % log_probs.shape[-1]

            # Now dimensions match: (batch_size, 1) + (batch_size, k)
            effective = (torch.arange(batch_size, device=src.device).unsqueeze(1) * num_beams + beam_indices).view(-1)
            beams = torch.cat([beams[effective], token_indices.view(-1, 1)], dim=1)
            beam_scores = top_scores.view(-1)
            finished_beams = finished_beams | (beams[:, -1] == tokenizer.eos_token_id)

        final_beams = beams.view(batch_size, num_beams, -1)
        best_beams = final_beams[:, 0, :]
        self.train()
        return best_beams

In [ ]:
def count_parameters(model):
    table_data = []
    total_params = 0
    trainable_params = 0

    # 1. Global Counts
    for p in model.parameters():
        total_params += p.numel()
        if p.requires_grad:
            trainable_params += p.numel()

    print("="*40)
    print(f"📊 MODEL STATISTICS")
    print("="*40)
    print(f"Total Parameters:     {total_params:,}  ({total_params/1e6:.2f}M)")
    print(f"Trainable Parameters: {trainable_params:,}  ({trainable_params/1e6:.2f}M)")
    print("-" * 40)

    # 2. Section Breakdown
    def get_params(module):
        return sum(p.numel() for p in module.parameters())

    if hasattr(model, 'encoder'):
        enc_p = get_params(model.encoder)
        print(f"  • Encoder (FNet):   {enc_p:,}  ({enc_p/1e6:.2f}M)")

    if hasattr(model, 'decoder'):
        dec_p = get_params(model.decoder)
        print(f"  • Decoder (RoPE):   {dec_p:,}  ({dec_p/1e6:.2f}M)")

    if hasattr(model, 'embedding'):
        emb_p = get_params(model.embedding)
        print(f"  • Embeddings:       {emb_p:,}  ({emb_p/1e6:.2f}M)")

    print("="*40)



## Functions (Loss, Eval etc)

In [ ]:

translation_loss_fn = nn.CrossEntropyLoss(
    ignore_index=-100,  # We don't calculate loss for pad tokens. Pad tokens are replaced with -100 by DataCollatorForSeq2Seq.
    label_smoothing=LABEL_SMOOTHING_EPSILON
)
def calculate_combined_loss(model_outputs, target_labels):
    """Calculates the loss based on the model's output structure."""
    logits = model_outputs
    translation_loss = translation_loss_fn(logits.reshape(-1, logits.shape[-1]), target_labels.reshape(-1))
    loss_dict = {'total': translation_loss.item()}
    return translation_loss, loss_dict

from torchmetrics.text import SacreBLEUScore

def evaluate(model, dataloader, device):
    # Use SacreBLEUScore (defaults to '13a' tokenizer, the WMT standard)
    metric = SacreBLEUScore().to(device)

    model.eval()

    # Use no_grad to save memory and speed up validation
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating", leave=False):
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels']

            # Generate predictions
            generated_ids = model.generate(input_ids, max_length=MAX_LENGTH, num_beams=5)

            # Decode predictions
            pred_texts = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

            # Decode labels (Fixing -100 padding)
            labels[labels == -100] = tokenizer.pad_token_id
            ref_texts = tokenizer.batch_decode(labels, skip_special_tokens=True)

            # Update Metric
            # SacreBLEU expects references as a list of lists: [[ref1], [ref2], ...]
            formatted_refs = [[ref] for ref in ref_texts]
            metric.update(pred_texts, formatted_refs)

    model.train()

    # Compute returns a tensor, .item() converts it to a standard python float
    return metric.compute().item()



## WARNING! THIS CAN'T BE USED FOR FNET
def generate_sample_translations(model, device, sentences_de):
    """Generates and prints sample translations using beam search."""
    print("\n--- Generating Sample Translations (with Beam Search) ---")
    orig_model = getattr(model, '_orig_mod', model)
    orig_model.eval()

    inputs = tokenizer(sentences_de, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH)
    input_ids = inputs.input_ids.to(device)
    generated_ids = orig_model.generate(input_ids, max_length=MAX_LENGTH, num_beams=5)

    translations = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    for src, out in zip(sentences_de, translations):
        print(f"  DE Source: {src}")
        print(f"  EN Output: {out}")
        print("-" * 20)
    orig_model.train()

sample_sentences_de_for_tracking = [
    "Eine Katze sitzt auf der Matte.",
    "Ein Mann in einem roten Hemd liest ein Buch.",
    "Was ist die Hauptstadt von Deutschland?",
    "Ich gehe ins Kino, weil der Film sehr gut ist.",
]

def init_other_linear_weights(m):
    if isinstance(m, nn.Linear):
        # The 'is not' check correctly skips the final_linear layer,
        # leaving its weights tied to the correctly initialized embeddings.
        if m is not getattr(model, '_orig_mod', model).final_linear:
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)




In [ ]:
import json
import os
import subprocess
import torch
import hashlib
import sys
import shutil

# This logger will be configured and used in the main training script
import logging
logger = logging.getLogger(__name__)


def log_to_run_specific_file(run_dir):
    run_log_path = os.path.join(run_dir, "run_log.txt")
    file_handler = logging.FileHandler(run_log_path)
    file_handler.setFormatter(logging.Formatter('%(asctime)s [%(levelname)s] %(message)s'))
    logger.addHandler(file_handler)
    return file_handler

def log_configurations(log_dir, config_vars):
    # (Same as your provided function)
    config_path = os.path.join(log_dir, "config.json")
    try:
        with open(config_path, 'w') as f:
            serializable_configs = {k: v for k, v in config_vars.items() if isinstance(v, (int, float, str, bool, list, dict, type(None)))}
            json.dump(serializable_configs, f, indent=4)
        logger.info(f"Configurations saved to {config_path}")
    except Exception as e:
        logger.error(f"Could not save configurations: {e}")

def log_environment(log_dir):
    # (Same as your provided function)
    env_path = os.path.join(log_dir, "environment.txt")
    try:
        with open(env_path, 'w') as f:
            f.write(f"--- Timestamp (UTC): {datetime.datetime.utcnow().isoformat()} ---\n")
            f.write(f"Python Version: {sys.version}\n")
            f.write(f"PyTorch Version: {torch.__version__}\n")
            f.write(f"CUDA Available: {torch.cuda.is_available()}\n")
            if torch.cuda.is_available():
                f.write(f"CUDA Version: {torch.version.cuda}\n")
                f.write(f"CuDNN Version: {torch.backends.cudnn.version()}\n")
                f.write(f"Number of GPUs: {torch.cuda.device_count()}\n")
                f.write(f"GPU Name: {torch.cuda.get_device_name(0)}\n")
            f.write("\n--- Full pip freeze ---\n")
            result = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=subprocess.PIPE, text=True, check=True)
            f.write(result.stdout)
        logger.info(f"Environment info saved to {env_path}")
    except Exception as e:
        logger.error(f"Could not save environment info: {e}")

def log_code_snapshot(log_dir, script_path):
    # NOTE: In Colab, you must save your notebook as a .py file for this to work.
    # For example, file -> "Save a copy as .py"
    code_dir = os.path.join(log_dir, "code_snapshot")
    os.makedirs(code_dir, exist_ok=True)
    if script_path and os.path.exists(script_path):
        try:
            shutil.copy(script_path, os.path.join(code_dir, os.path.basename(script_path)))
            logger.info(f"Copied script '{script_path}' to snapshot directory for verification.")
        except Exception as e:
            logger.error(f"Could not copy script for snapshot: {e}")
    else:
        logger.warning(f"Code Snapshot: Script path '{script_path}' not found. SKIPPING.")

def get_file_hash(filepath):
    # (Same as your provided function)
    sha256_hash = hashlib.sha256()
    try:
        with open(filepath, "rb") as f:
            for byte_block in iter(lambda: f.read(4096), b""):
                sha256_hash.update(byte_block)
        return sha256_hash.hexdigest()
    except Exception as e:
        logger.error(f"Could not generate hash for {filepath}: {e}")
        return None

def create_checksum_file(run_dir, artifacts_dict):
    checksum_file_path = os.path.join(run_dir, "checksums.sha256")
    logger.info(f"--- Creating digital fingerprints for key artifacts ---")
    with open(checksum_file_path, "w") as f:
        f.write(f"SHA256 Checksums for run: {os.path.basename(run_dir)}\n")
        for name, path in artifacts_dict.items():
            if path and os.path.exists(path):
                file_hash = get_file_hash(path)
                if file_hash:
                    log_message = f"  - {name} ({os.path.basename(path)}): {file_hash}"
                    logger.info(log_message)
                    f.write(f"{file_hash}  {os.path.basename(path)}\n")
            else:
                logger.warning(f"  - Skipped hashing '{name}', file not found: {path}")
    logger.info(f"Checksums saved to {checksum_file_path}")

def init_weights_kaiming(m):
    """
    Applies Kaiming He initialization to Linear layers.
    This is the standard, superior way to initialize deep Transformers.
    NOTE: We will handle the Embedding layer separately.
    """

    if isinstance(m, nn.Linear):
        nn.init.kaiming_uniform_(m.weight, a=math.sqrt(5)) # a=sqrt(5) mimics default PyTorch for LeakyReLU
        if m.bias is not None:
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(m.weight)
            bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
            nn.init.uniform_(m.bias, -bound, bound)


def init_weights_fnet(m):
    """
    Specific initialization for FNet Hybrid.
    FNet is essentially a BERT-like encoder, so we use BERT-style initialization
    (Truncated Normal or Xavier) rather than Kaiming.
    """
    if isinstance(m, nn.Linear):
        # Xavier (Glorot) Uniform is the standard for Transformer/FNet attention/FFN layers
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

    elif isinstance(m, nn.Embedding):
        # Critical: Keep embedding variance low (0.02)
        nn.init.normal_(m.weight, mean=0.0, std=0.02)

    # Handle the RMSNorms if they have learnable parameters
    elif isinstance(m, (nn.LayerNorm, RMSNorm)):
        if hasattr(m, 'weight') and m.weight is not None:
            nn.init.ones_(m.weight)
        if hasattr(m, 'bias') and m.bias is not None:
            nn.init.zeros_(m.bias)



## Training Loop

In [ ]:
if __name__ == '__main__':

    experiment_name = f"{MODEL_CHOICE}"
    CURRENT_RUN_DIR = os.path.join(DRIVE_BASE_PATH, experiment_name)
    SAVE_DIR = os.path.join(CURRENT_RUN_DIR, "models")
    LOG_DIR_TENSORBOARD = os.path.join(CURRENT_RUN_DIR, "tensorboard_logs")
    LOG_FILE_TXT = os.path.join(CURRENT_RUN_DIR, "run_log.txt")

    os.makedirs(SAVE_DIR, exist_ok=True)
    os.makedirs(LOG_DIR_TENSORBOARD, exist_ok=True)

    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s [%(levelname)s] %(message)s',
        handlers=[logging.FileHandler(LOG_FILE_TXT), logging.StreamHandler(sys.stdout)],
        force=True
    )
    logger = logging.getLogger(__name__)
    writer = SummaryWriter(LOG_DIR_TENSORBOARD)

    logger.info(f"--- LAUNCHING EXPERIMENT: {experiment_name} ---")

    all_configs = {k: v for k, v in globals().items() if k.isupper()}
    log_configurations(CURRENT_RUN_DIR, all_configs)
    log_environment(CURRENT_RUN_DIR)

    logger.info(f"--- Initializing FNetHybridTransformer ---")
    model = FNetHybridTransformer(
        num_encoder_layers=NUM_ENCODER_LAYERS,
        num_decoder_layers=NUM_DECODER_LAYERS,
        num_heads=NUM_HEADS,
        d_model=D_MODEL,
        dff=D_FF,
        vocab_size=VOCAB_SIZE,
        max_length=MAX_LENGTH,
        dropout=DROPOUT
    )

    model.apply(init_weights_fnet)
    nn.init.normal_(model.pos_embedding.weight, mean=0.0, std=0.02)
    model.final_linear.weight = model.embedding.weight

    model.to(device)
    count_parameters(model)

    # 4. SETUP OPTIMIZER
    optimizer = torch.optim.AdamW(model.parameters(), lr=PEAK_LEARNING_RATE, betas=(0.9, 0.98),
                                  eps=1e-9, weight_decay=WEIGHT_DECAY)

    # Scheduler
    scheduler = get_cosine_schedule_with_warmup(optimizer=optimizer, num_warmup_steps=WARMUP_STEPS,
                                                num_training_steps=TARGET_TRAINING_STEPS)
    scaler = torch.cuda.amp.GradScaler()

# --- AUTO-RESUME LOGIC (SMARTER VERSION) ---
    global_step = 0
    best_bleu = 0.0
    LAST_CHECKPOINT_PATH = os.path.join(SAVE_DIR, "last.pt")
    BEST_CHECKPOINT_PATH = os.path.join(SAVE_DIR, "best.pt")

    # 1. Try to find the latest checkpoint (if it exists)
    if os.path.exists(LAST_CHECKPOINT_PATH):
        logger.info(f"🔄 Found checkpoint at {LAST_CHECKPOINT_PATH}. Resuming...")
        checkpoint = torch.load(LAST_CHECKPOINT_PATH, map_location=device)

        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        scaler.load_state_dict(checkpoint['scaler_state_dict'])

        global_step = checkpoint['global_step']
        best_bleu = checkpoint.get('best_bleu', 0.0)
        logger.info(f"   ✅ Resumed from Step {global_step} (LAST)")

    # 2. If no LAST, try to find the BEST checkpoint (Fall back to this!)
    elif os.path.exists(BEST_CHECKPOINT_PATH):
        logger.info(f"🔙 'last.pt' not found. Falling back to BEST checkpoint: {BEST_CHECKPOINT_PATH}")
        checkpoint = torch.load(BEST_CHECKPOINT_PATH, map_location=device)

        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        scaler.load_state_dict(checkpoint['scaler_state_dict'])

        global_step = checkpoint['global_step']
        best_bleu = checkpoint.get('best_bleu', 0.0)
        logger.info(f"   ✅ Resumed from Step {global_step} (BEST)")

    # 3. Start Fresh
    else:
        logger.info("🆕 No checkpoint found. Starting fresh training.")
    # 5. TRAINING LOOP
    model.train()

    # Resume progress bar from global_step
    progress_bar = tqdm(total=TARGET_TRAINING_STEPS, initial=global_step, desc="Training Steps")
    training_complete = False

    # Initialize gradients
    optimizer.zero_grad(set_to_none=True)

    # We iterate until global_step reaches the target
    epoch = 0
    while not training_complete:
        train_dataloader.generator.manual_seed(SEED + epoch)
        epoch += 1

        for batch_idx, batch in enumerate(train_dataloader):
            if global_step >= TARGET_TRAINING_STEPS:
                training_complete = True
                break

            input_ids = batch['input_ids'].to(device, non_blocking=True)
            labels = batch['labels'].to(device, non_blocking=True)

            decoder_start_token = torch.full((labels.shape[0], 1), tokenizer.pad_token_id, dtype=torch.long, device=device)
            decoder_input_ids = torch.cat([decoder_start_token, labels[:, :-1]], dim=1)
            decoder_input_ids[decoder_input_ids == -100] = tokenizer.pad_token_id
            target_labels = labels

            src_padding_mask, tgt_padding_mask, mem_key_padding_mask, tgt_mask = model.create_masks(input_ids, decoder_input_ids)
            tgt_padding_mask[:, 0] = False

            with torch.autocast(device_type="cuda", dtype=torch.float16):
                model_outputs = model(src=input_ids, tgt=decoder_input_ids, src_padding_mask=src_padding_mask,
                                      tgt_padding_mask=tgt_padding_mask, memory_key_padding_mask=mem_key_padding_mask,
                                      tgt_mask=tgt_mask)
                loss, loss_components = calculate_combined_loss(model_outputs, target_labels)

                # --- GRADIENT ACCUMULATION SCALING ---
                loss = loss / GRAD_ACCUMULATION_STEPS

            # Accumulate gradients (no optimizer step yet)
            scaler.scale(loss).backward()

            # --- OPTIMIZER STEP (Conditional) ---
            if (batch_idx + 1) % GRAD_ACCUMULATION_STEPS == 0:
                scaler.unscale_(optimizer)
                total_grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

                scaler.step(optimizer)
                scaler.update()
                scheduler.step()

                # Reset gradients
                optimizer.zero_grad(set_to_none=True)

                global_step += 1
                progress_bar.update(1)
                lr = scheduler.get_last_lr()[0]

                if global_step % 20 == 0:
                    # Scale loss back up for logging purposes
                    logged_loss = loss.item() * GRAD_ACCUMULATION_STEPS
                    writer.add_scalar('train/loss', logged_loss, global_step)
                    writer.add_scalar('train/learning_rate', lr, global_step)
                    writer.add_scalar('train/gradient_norm', total_grad_norm.item(), global_step)
                    progress_bar.set_postfix(
                        loss=f"{logged_loss:.2f}",
                        lr=f"{lr:.2e}",
                        grad=f"{total_grad_norm.item():.2f}"  # Showing Gradients
                    )

                # --- PERIODIC SAVING (Every 500 Steps) ---
                # Saves you if Colab crashes mid-epoch
                if global_step % 500 == 0:
                     torch.save({
                        'global_step': global_step,
                        'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'scheduler_state_dict': scheduler.state_dict(),
                        'scaler_state_dict': scaler.state_dict(),
                        'best_bleu': best_bleu
                    }, LAST_CHECKPOINT_PATH)

                # --- VALIDATION CHECK ---
                if global_step in VALIDATION_SCHEDULE:
                    logger.info(f"\n--- Validation at Step {global_step} ---")
                    bleu_score = evaluate(model, val_dataloader, device)
                    writer.add_scalar('validation/bleu', bleu_score, global_step)
                    logger.info(f"Validation BLEU: {bleu_score:.4f} (Best: {best_bleu:.4f})")
                    #generate_sample_translations(model, device, sample_sentences_de_for_tracking)

                    if bleu_score > best_bleu:
                        best_bleu = bleu_score
                        logger.info(f"  New best BLEU! Saving best model...")
                        # Save EVERYTHING so you can resume even from best model
                        torch.save({
                            'global_step': global_step,
                            'model_state_dict': model.state_dict(),
                            'optimizer_state_dict': optimizer.state_dict(),
                            'scheduler_state_dict': scheduler.state_dict(),
                            'scaler_state_dict': scaler.state_dict(),
                            'best_bleu': best_bleu
                        }, BEST_CHECKPOINT_PATH)

                    model.train()

    progress_bar.close()
    writer.close()

    # Save Final (With States)
    torch.save({
        'global_step': global_step,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'best_bleu': best_bleu
    }, LAST_CHECKPOINT_PATH)

    print("\n" + "*"*80)
    print(" EXPERIMENT COMPLETE ")
    print("*"*80)

In [ ]:
import os
import sys
import torch
import transformers
import datasets
import torchmetrics
import numpy
import pkg_resources

def log_environment_separate(log_dir):
    # Define the separate file path
    meta_file = os.path.join(log_dir, "system_metadata.txt")

    with open(meta_file, "w") as f:
        # --- PART 1: SUMMARY ---
        f.write("="*40 + "\n")
        f.write("CORE ENVIRONMENT SUMMARY\n")
        f.write("="*40 + "\n")
        f.write(f"Python:       {sys.version.split()[0]}\n")
        f.write(f"PyTorch:      {torch.__version__}\n")
        f.write(f"Transformers: {transformers.__version__}\n")
        f.write(f"Datasets:     {datasets.__version__}\n")
        f.write(f"TorchMetrics: {torchmetrics.__version__}\n")
        f.write(f"NumPy:        {numpy.__version__}\n")

        try:
            import sacrebleu
            f.write(f"SacreBLEU:    {sacrebleu.__version__}\n")
        except ImportError:
            f.write("SacreBLEU:    Not Installed\n")

        if torch.cuda.is_available():
            f.write(f"GPU Name:     {torch.cuda.get_device_name(0)}\n")
            f.write(f"CUDA Ver:     {torch.version.cuda}\n")
            f.write(f"Capability:   {torch.cuda.get_device_capability(0)}\n")
        else:
            f.write("GPU:          None (CPU Only)\n")

        # --- PART 2: FULL FREEZE ---
        f.write("\n" + "="*40 + "\n")
        f.write("FULL LIBRARY DEPENDENCIES (PIP FREEZE)\n")
        f.write("="*40 + "\n")

        installed_packages = {d.project_name: d.version for d in pkg_resources.working_set}
        for package, version in sorted(installed_packages.items()):
            f.write(f"{package}=={version}\n")

    print(f"✅ Environment details saved SEPARATELY to: {meta_file}")

# Execute
# Assumes CURRENT_RUN_DIR is defined from your config
log_environment_separate(CURRENT_RUN_DIR)

In [ ]:
# TENSORBOARD VISUALIZATION

%load_ext tensorboard

TENSORBOARD_BASE_DIR = os.path.join(DRIVE_BASE_PATH)

%tensorboard --logdir "{TENSORBOARD_BASE_DIR}"

In [ ]:
from google.colab import runtime
runtime.unassign()

## End